In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor ,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense,BatchNormalization,Dropout

In [28]:
df = pd.read_csv("synthetic_food_delivery_transactions.csv")
df.head()

,order_id,customer_id,customer_loyalty_tier,customer_acquisition_channel,customer_signup_date,restaurant_id,restaurant_cuisine,restaurant_city,restaurant_rating,restaurant_avg_prep_time,...,total_amount,discount_code,discount_type,discount_value,order_status,weather_temperature,weather_precipitation,is_weekend,is_holiday,event_flag
0,4002,925,Gold,social_media,2024-07-22,124,American,Los Angeles,3.6,23,...,25.42,PROMO-BFUZ,percentage,4.97,completed,2.9,25.6,True,True,NaN
1,4879,3163,Bronze,referral,2023-03-17,43,Chinese,Houston,4.8,32,...,22.70,NaN,NaN,0.00,completed,4.0,0.4,True,True,NaN
2,1124,4154,Bronze,referral,2023-01-14,188,American,San Antonio,3.4,11,...,27.74,NaN,NaN,0.00,completed,4.4,0.0,True,True,NaN
3,6764,4530,Bronze,email,2024-02-26,91,Mediterranean,New York,3.1,17,...,31.85,NaN,NaN,0.00,completed,9.9,0.0,True,True,NaN
4,9073,2570,Bronze,paid_ads,2024-08-25,84,Mediterranean,San Antonio,3.2,28,...,28.46,NaN,NaN,0.00,completed,7.0,16.1,True,True,NaN


In [29]:
df.columns

Index(['order_id', 'customer_id', 'customer_loyalty_tier',
       'customer_acquisition_channel', 'customer_signup_date', 'restaurant_id',
       'restaurant_cuisine', 'restaurant_city', 'restaurant_rating',
       'restaurant_avg_prep_time', 'menu_item_id', 'menu_item_name',
       'menu_item_category', 'item_price', 'quantity', 'order_timestamp',
       'delivery_timestamp', 'delivery_duration_actual',
       'delivery_duration_estimated', 'delivery_fee', 'tip', 'total_amount',
       'discount_code', 'discount_type', 'discount_value', 'order_status',
       'weather_temperature', 'weather_precipitation', 'is_weekend',
       'is_holiday', 'event_flag'],
      dtype='object')

In [30]:
df["delivery_duration_actual"]

0       59.0
1       57.0
2       41.0
3       50.0
4       69.0
        ... 
9995    42.0
9996     NaN
9997    68.0
9998    68.0
9999    67.0
Name: delivery_duration_actual, Length: 10000, dtype: float64

In [31]:
df.shape

(10000, 31)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   order_id                      10000 non-null  int64  
 1   customer_id                   10000 non-null  int64  
 2   customer_loyalty_tier         10000 non-null  object 
 3   customer_acquisition_channel  10000 non-null  object 
 4   customer_signup_date          10000 non-null  object 
 5   restaurant_id                 10000 non-null  int64  
 6   restaurant_cuisine            10000 non-null  object 
 7   restaurant_city               10000 non-null  object 
 8   restaurant_rating             10000 non-null  float64
 9   restaurant_avg_prep_time      10000 non-null  int64  
 10  menu_item_id                  10000 non-null  int64  
 11  menu_item_name                10000 non-null  object 
 12  menu_item_category            10000 non-null  object 
 13  it

In [33]:
col = ["restaurant_rating", "restaurant_avg_prep_time", "item_price", "quantity", "delivery_duration_estimated", "delivery_fee", "tip", "total_amount", "discount_value", "weather_temperature", "weather_precipitation","delivery_duration_actual","customer_loyalty_tier","customer_acquisition_channel","restaurant_cuisine","order_timestamp","restaurant_city","is_holiday","is_weekend"]

dro_col = df.drop(columns=col,axis=1).columns

df.drop(columns=dro_col,axis=1,inplace=True)

df.head()

,customer_loyalty_tier,customer_acquisition_channel,restaurant_cuisine,restaurant_city,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,discount_value,weather_temperature,weather_precipitation,is_weekend,is_holiday
0,Gold,social_media,American,Los Angeles,3.6,23,24.86,1,2023-01-01 00:10:17,59.0,48,3.48,2.05,25.42,4.97,2.9,25.6,True,True
1,Bronze,referral,Chinese,Houston,4.8,32,15.25,1,2023-01-01 00:29:15,57.0,54,5.24,2.21,22.70,0.00,4.0,0.4,True,True
2,Bronze,referral,American,San Antonio,3.4,11,18.50,1,2023-01-01 04:06:27,41.0,29,5.95,3.29,27.74,0.00,4.4,0.0,True,True
3,Bronze,email,Mediterranean,New York,3.1,17,24.51,1,2023-01-01 05:05:28,50.0,42,7.27,0.07,31.85,0.00,9.9,0.0,True,True
4,Bronze,paid_ads,Mediterranean,San Antonio,3.2,28,23.50,1,2023-01-01 08:46:51,69.0,52,4.06,0.90,28.46,0.00,7.0,16.1,True,True


In [34]:
df.duplicated().sum()

np.int64(0)

In [35]:
# df["discount_type"].mode()

In [36]:
df["delivery_duration_actual"] = df["delivery_duration_actual"].fillna(df["delivery_duration_actual"].mean())
# df["discount_type"] = df["discount_type"].fillna("percentage")

In [37]:
df["order_timestamp"]

def houer(times):
    time = str(times)
    new_time = time[11:len(time)-6]
    return int(new_time)

df["order_timestamp"] = df["order_timestamp"].apply(houer)
df.head()

,customer_loyalty_tier,customer_acquisition_channel,restaurant_cuisine,restaurant_city,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,discount_value,weather_temperature,weather_precipitation,is_weekend,is_holiday
0,Gold,social_media,American,Los Angeles,3.6,23,24.86,1,0,59.0,48,3.48,2.05,25.42,4.97,2.9,25.6,True,True
1,Bronze,referral,Chinese,Houston,4.8,32,15.25,1,0,57.0,54,5.24,2.21,22.70,0.00,4.0,0.4,True,True
2,Bronze,referral,American,San Antonio,3.4,11,18.50,1,4,41.0,29,5.95,3.29,27.74,0.00,4.4,0.0,True,True
3,Bronze,email,Mediterranean,New York,3.1,17,24.51,1,5,50.0,42,7.27,0.07,31.85,0.00,9.9,0.0,True,True
4,Bronze,paid_ads,Mediterranean,San Antonio,3.2,28,23.50,1,8,69.0,52,4.06,0.90,28.46,0.00,7.0,16.1,True,True


In [38]:
df.head()

,customer_loyalty_tier,customer_acquisition_channel,restaurant_cuisine,restaurant_city,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,discount_value,weather_temperature,weather_precipitation,is_weekend,is_holiday
0,Gold,social_media,American,Los Angeles,3.6,23,24.86,1,0,59.0,48,3.48,2.05,25.42,4.97,2.9,25.6,True,True
1,Bronze,referral,Chinese,Houston,4.8,32,15.25,1,0,57.0,54,5.24,2.21,22.70,0.00,4.0,0.4,True,True
2,Bronze,referral,American,San Antonio,3.4,11,18.50,1,4,41.0,29,5.95,3.29,27.74,0.00,4.4,0.0,True,True
3,Bronze,email,Mediterranean,New York,3.1,17,24.51,1,5,50.0,42,7.27,0.07,31.85,0.00,9.9,0.0,True,True
4,Bronze,paid_ads,Mediterranean,San Antonio,3.2,28,23.50,1,8,69.0,52,4.06,0.90,28.46,0.00,7.0,16.1,True,True


In [39]:
bool_col = ["is_weekend","is_holiday"]

df[bool_col] = df[bool_col].astype(int)
df.head()

,customer_loyalty_tier,customer_acquisition_channel,restaurant_cuisine,restaurant_city,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,discount_value,weather_temperature,weather_precipitation,is_weekend,is_holiday
0,Gold,social_media,American,Los Angeles,3.6,23,24.86,1,0,59.0,48,3.48,2.05,25.42,4.97,2.9,25.6,1,1
1,Bronze,referral,Chinese,Houston,4.8,32,15.25,1,0,57.0,54,5.24,2.21,22.70,0.00,4.0,0.4,1,1
2,Bronze,referral,American,San Antonio,3.4,11,18.50,1,4,41.0,29,5.95,3.29,27.74,0.00,4.4,0.0,1,1
3,Bronze,email,Mediterranean,New York,3.1,17,24.51,1,5,50.0,42,7.27,0.07,31.85,0.00,9.9,0.0,1,1
4,Bronze,paid_ads,Mediterranean,San Antonio,3.2,28,23.50,1,8,69.0,52,4.06,0.90,28.46,0.00,7.0,16.1,1,1


In [40]:
print(df["customer_loyalty_tier"].unique())
print(df["customer_acquisition_channel"].unique())
print(df["restaurant_cuisine"].unique())
print(df["restaurant_city"].unique())

['Gold' 'Bronze' 'Silver']
['social_media' 'referral' 'email' 'paid_ads' 'organic']
['American' 'Chinese' 'Mediterranean' 'Fast Food' 'Indian' 'Italian'
 'Japanese' 'Mexican']
['Los Angeles' 'Houston' 'San Antonio' 'New York' 'San Diego' 'Chicago'
 'Phoenix' 'Philadelphia']


In [41]:
one_col = ["customer_loyalty_tier","customer_acquisition_channel","restaurant_cuisine","restaurant_city"]

df = pd.get_dummies(df,columns=one_col,drop_first=True,dtype=int)
df.head()

,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,...,restaurant_cuisine_Japanese,restaurant_cuisine_Mediterranean,restaurant_cuisine_Mexican,restaurant_city_Houston,restaurant_city_Los Angeles,restaurant_city_New York,restaurant_city_Philadelphia,restaurant_city_Phoenix,restaurant_city_San Antonio,restaurant_city_San Diego
0,3.6,23,24.86,1,0,59.0,48,3.48,2.05,25.42,...,0,0,0,0,1,0,0,0,0,0
1,4.8,32,15.25,1,0,57.0,54,5.24,2.21,22.70,...,0,0,0,1,0,0,0,0,0,0
2,3.4,11,18.50,1,4,41.0,29,5.95,3.29,27.74,...,0,0,0,0,0,0,0,0,1,0
3,3.1,17,24.51,1,5,50.0,42,7.27,0.07,31.85,...,0,1,0,0,0,1,0,0,0,0
4,3.2,28,23.50,1,8,69.0,52,4.06,0.90,28.46,...,0,1,0,0,0,0,0,0,1,0


In [42]:
scaler_col = ["restaurant_rating", "restaurant_avg_prep_time", "item_price", "quantity", "delivery_duration_estimated", "delivery_fee", "tip", "total_amount", "discount_value", "weather_temperature", "weather_precipitation","order_timestamp"]

scaler = StandardScaler()
df[scaler_col] = scaler.fit_transform(df[scaler_col])
df.head()

,restaurant_rating,restaurant_avg_prep_time,item_price,quantity,order_timestamp,delivery_duration_actual,delivery_duration_estimated,delivery_fee,tip,total_amount,...,restaurant_cuisine_Japanese,restaurant_cuisine_Mediterranean,restaurant_cuisine_Mexican,restaurant_city_Houston,restaurant_city_Los Angeles,restaurant_city_New York,restaurant_city_Philadelphia,restaurant_city_Phoenix,restaurant_city_San Antonio,restaurant_city_San Diego
0,-0.701994,0.101703,0.555208,-0.512142,-1.640487,59.0,0.673445,-0.888848,-0.398601,-0.437108,...,0,0,0,0,1,0,0,0,0,0
1,1.348471,1.370520,-0.574125,-0.512142,-1.640487,57.0,1.325408,0.127059,-0.353313,-0.557622,...,0,0,0,1,0,0,0,0,0,0
2,-1.043738,-1.590053,-0.192197,-0.512142,-1.064539,41.0,-1.391104,0.536885,-0.047625,-0.334317,...,0,0,0,0,0,0,0,0,1,0
3,-1.556354,-0.744175,0.514077,-0.512142,-0.920552,50.0,0.021482,1.298815,-0.959029,-0.152217,...,0,1,0,0,0,1,0,0,0,0
4,-1.385482,0.806601,0.395385,-0.512142,-0.488591,69.0,1.108087,-0.554060,-0.724102,-0.302416,...,0,1,0,0,0,0,0,0,1,0


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   restaurant_rating                          10000 non-null  float64
 1   restaurant_avg_prep_time                   10000 non-null  float64
 2   item_price                                 10000 non-null  float64
 3   quantity                                   10000 non-null  float64
 4   order_timestamp                            10000 non-null  float64
 5   delivery_duration_actual                   10000 non-null  float64
 6   delivery_duration_estimated                10000 non-null  float64
 7   delivery_fee                               10000 non-null  float64
 8   tip                                        10000 non-null  float64
 9   total_amount                               10000 non-null  float64
 10  discount_value         

In [44]:
X = df.drop("delivery_duration_actual",axis=1)
y = df["delivery_duration_actual"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [45]:
X.columns

Index(['restaurant_rating', 'restaurant_avg_prep_time', 'item_price',
       'quantity', 'order_timestamp', 'delivery_duration_estimated',
       'delivery_fee', 'tip', 'total_amount', 'discount_value',
       'weather_temperature', 'weather_precipitation', 'is_weekend',
       'is_holiday', 'customer_loyalty_tier_Gold',
       'customer_loyalty_tier_Silver', 'customer_acquisition_channel_organic',
       'customer_acquisition_channel_paid_ads',
       'customer_acquisition_channel_referral',
       'customer_acquisition_channel_social_media',
       'restaurant_cuisine_Chinese', 'restaurant_cuisine_Fast Food',
       'restaurant_cuisine_Indian', 'restaurant_cuisine_Italian',
       'restaurant_cuisine_Japanese', 'restaurant_cuisine_Mediterranean',
       'restaurant_cuisine_Mexican', 'restaurant_city_Houston',
       'restaurant_city_Los Angeles', 'restaurant_city_New York',
       'restaurant_city_Philadelphia', 'restaurant_city_Phoenix',
       'restaurant_city_San Antonio', 'restau

In [46]:
model_ann = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(128,activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64,activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32,activation="relu"),
    BatchNormalization(),
    Dropout(0.1),
    Dense(16,activation="relu"),
    BatchNormalization(),
    Dense(8,activation="relu"),
    Dense(1)
])

In [47]:
model_ann.compile(optimizer="adam",loss="mse",metrics=["r2_score"])

In [48]:
his = model_ann.fit(
    X_train,y_train,
    validation_data = (X_test,y_test),
    epochs = 100,batch_size = 16,
    verbose = 1
)

Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 1216.4191 - r2_score: -8.7482 - val_loss: 89.9330 - val_r2_score: 0.2793
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 96.0898 - r2_score: 0.2300 - val_loss: 50.8654 - val_r2_score: 0.5924
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 78.2934 - r2_score: 0.3726 - val_loss: 49.8204 - val_r2_score: 0.6008
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 72.5227 - r2_score: 0.4188 - val_loss: 47.2584 - val_r2_score: 0.6213
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 68.8373 - r2_score: 0.4483 - val_loss: 45.7573 - val_r2_score: 0.6333
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 62.9685 - r2_score: 0.4954 - val_loss: 45.3627 - val_r2_score: 0.6365
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 59.5505 - r2_score: 0.5228 - val_loss: 45.0566 - val_r2_score: 0.6389
Epoch 8/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 58.5471 - r2_score:

In [49]:
model_ann.evaluate(
    # X_train,y_train,
    X_test,y_test,
    verbose = 1
)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 43.1767 - r2_score: 0.6540  


[43.176734924316406, 0.6539902687072754]

In [50]:
results = []
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'K-Neighbors': KNeighborsRegressor(),
    'Support Vector Regressor': SVR(kernel='rbf'),
    "XGBoost": XGBRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
}

In [51]:
for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    R2_score = r2_score(y_test,y_pred)
    MSE = mean_squared_error(y_test, y_pred)
    MAE = mean_absolute_error(y_test, y_pred)
    RMSE = np.sqrt(MSE)
    results.append({
        "Name":name,
        "R2_score":R2_score,
        "MSE":MSE,
        "MAE":MAE,
        "RMSE":RMSE
    })

results_df = pd.DataFrame(results)
results_df

,Name,R2_score,MSE,MAE,RMSE
0,Linear Regression,0.625375,46.748309,5.658460,6.837273
1,Ridge Regression,0.625385,46.747135,5.658391,6.837188
2,Lasso Regression,0.626842,46.565241,5.647578,6.823873
3,ElasticNet,0.626406,46.619720,5.652129,6.827864
4,Decision Tree,0.337445,82.678275,7.282976,9.092759
5,Random Forest,0.666328,41.637900,5.369307,6.452744
6,Extra Trees,0.648906,43.812016,5.482445,6.619065
7,Gradient Boosting,0.678386,40.133207,5.302212,6.335078
8,K-Neighbors,0.492174,63.370130,6.429585,7.960536
9,Support Vector Regressor,0.625386,46.746913,5.650198,6.837171
